In [1]:
import importlib.util
import numpy as np
import torch
from pathlib import Path
from Bio.PDB import PDBParser

MODEL = "12_04052026"

_spec = importlib.util.spec_from_file_location('pocket_model', f'./models/{MODEL}.py')
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

PocketFinder                = _mod.PocketFinder
AA_TO_IDX                   = _mod.AA_TO_IDX
PHYSCHEM                    = _mod.PHYSCHEM
compute_structural_features = _mod.compute_structural_features
compute_hse                 = _mod.compute_hse

In [2]:
CHECKPOINT = f'./models/{MODEL}.pt'

state    = torch.load(CHECKPOINT, map_location='cpu')
in_dim   = state['embedding.weight'].shape[1]
hidden   = state['embedding.weight'].shape[0]
n_layers = sum(1 for k in state if k.startswith('convs.') and k.endswith('.eps'))

model = PocketFinder(in_dim=in_dim, hidden=hidden, n_layers=n_layers)
model.load_state_dict(state)
model.eval()
print(f'loaded: in_dim={in_dim}  hidden={hidden}  n_layers={n_layers}  params={sum(p.numel() for p in model.parameters())}')

loaded: in_dim=18  hidden=256  n_layers=4  params=541505


In [3]:
_pdb_parser = PDBParser(QUIET=True)


def parse_pdb_protein(path, chain_id=None):
    structure = _pdb_parser.get_structure('p', str(path))
    coords, residue_names, res_ids = [], [], []
    for residue in structure.get_residues():
        if residue.get_id()[0] != ' ':
            continue
        chain = residue.get_parent().get_id()
        if chain_id is not None and chain != chain_id:
            continue
        res_name = residue.get_resname()
        if res_name not in AA_TO_IDX or 'CA' not in residue:
            continue
        coords.append(residue['CA'].get_vector().get_array())
        residue_names.append(res_name)
        res_ids.append(f"{chain}:{res_name}{residue.get_id()[1]}")

    coords     = np.array(coords, dtype=np.float32)
    physchem   = np.array([PHYSCHEM[r] for r in residue_names], dtype=np.float32)
    structural = compute_structural_features(coords)
    hse        = compute_hse(coords)

    def norm(x):
        std = x.std(axis=0)
        std[std < 1e-8] = 1.0
        return (x - x.mean(axis=0)) / std

    h = np.concatenate([norm(physchem), norm(structural), norm(hse)], axis=1)
    return torch.tensor(coords, dtype=torch.float32), torch.tensor(h, dtype=torch.float32), res_ids


def parse_heavy_atoms(path, chain_id=None):
    structure = _pdb_parser.get_structure('p', str(path))
    all_coords = []
    res_atoms  = {}
    for residue in structure.get_residues():
        if residue.get_id()[0] != ' ':
            continue
        chain = residue.get_parent().get_id()
        if chain_id is not None and chain != chain_id:
            continue
        if residue.get_resname() not in AA_TO_IDX:
            continue
        atoms = [a.get_vector().get_array() for a in residue.get_atoms()
                 if a.element != 'H']
        if not atoms:
            continue
        res_id = f"{chain}:{residue.get_resname()}{residue.get_id()[1]}"
        res_atoms[res_id] = np.array(atoms, dtype=np.float32)
        all_coords.extend(atoms)
    return np.array(all_coords, dtype=np.float32), res_atoms

In [4]:
def predict_pocket(protein_path, threshold=0.5, chain_id=None):
    pos, h, res_ids = parse_pdb_protein(protein_path, chain_id=chain_id)
    batch = torch.zeros(len(pos), dtype=torch.long)

    with torch.no_grad():
        probs = model(h, pos, batch).sigmoid()

    mask           = probs > threshold
    pocket_indices = mask.nonzero(as_tuple=True)[0].tolist()
    pocket_res_ids = {res_ids[i] for i in pocket_indices}

    protein_heavy, res_atoms = parse_heavy_atoms(protein_path, chain_id=chain_id)
    pocket_atoms     = [res_atoms[r] for r in pocket_res_ids if r in res_atoms]
    non_pocket_atoms = [res_atoms[r] for r in res_atoms if r not in pocket_res_ids]
    pocket_heavy     = np.concatenate(pocket_atoms)     if pocket_atoms     else np.zeros((0, 3), dtype=np.float32)
    non_pocket_heavy = np.concatenate(non_pocket_atoms) if non_pocket_atoms else np.zeros((0, 3), dtype=np.float32)

    return {
        'probs':            probs,
        'pocket_indices':   pocket_indices,
        'pocket_res_ids':   sorted(pocket_res_ids),
        'pocket_heavy':     pocket_heavy,
        'non_pocket_heavy': non_pocket_heavy,
        'protein_heavy':    protein_heavy,
        'all_coords':       pos,
        'res_ids':          res_ids,
    }

In [5]:
COACH420_DIR = Path('./data/p2rank-datasets/coach420')
PROTEIN = "2qzzA"  # change to any coach420 protein
PROTEIN_PATH = COACH420_DIR / f'{PROTEIN}.pdb'

result = predict_pocket(PROTEIN_PATH, threshold=0.6)

print(f"total residues     : {len(result['res_ids'])}")
print(f"pocket residues    : {len(result['pocket_indices'])} ({100*len(result['pocket_indices'])/len(result['res_ids']):.1f}%)")
print(f"pocket heavy atoms : {len(result['pocket_heavy'])}")
print(f"protein heavy atoms: {len(result['protein_heavy'])}")
print(f"prob range         : [{result['probs'].min().item():.3f}, {result['probs'].max().item():.3f}]")

total residues     : 310
pocket residues    : 65 (21.0%)
pocket heavy atoms : 507
protein heavy atoms: 2477
prob range         : [0.000, 0.954]


In [6]:
K = 20
top_k = result['probs'].topk(K)
print(f'top-{K} pocket residues:')
for prob, idx in zip(top_k.values.tolist(), top_k.indices.tolist()):
    print(f'  {result["res_ids"][idx]:>12s}  p={prob:.3f}')

top-20 pocket residues:
      A:LYS128  p=0.954
      A:GLY109  p=0.921
      A:LEU261  p=0.921
       A:GLY16  p=0.920
       A:THR34  p=0.920
      A:TYR153  p=0.916
       A:PHE33  p=0.915
      A:CYS149  p=0.914
       A:GLU58  p=0.914
       A:GLY57  p=0.905
      A:PHE310  p=0.904
       A:ARG35  p=0.900
       A:THR12  p=0.898
       A:ILE15  p=0.888
       A:GLY13  p=0.885
       A:TYR14  p=0.885
      A:VAL110  p=0.872
      A:PHE154  p=0.866
      A:SER307  p=0.865
       A:PRO36  p=0.863


In [11]:
from scipy.spatial import cKDTree

def pocket_to_grid(pocket_coords, protein_coords, non_pocket_coords, spacing=1.0, padding=6.0, max_dist_from_pocket=6.0, voronoi_ratio=1.0):
    bb_min = pocket_coords.min(0) - padding
    bb_max = pocket_coords.max(0) + padding
    grid_shape = np.ceil((bb_max - bb_min) / spacing).astype(int)

    xi = np.arange(grid_shape[0]) * spacing + bb_min[0]
    yi = np.arange(grid_shape[1]) * spacing + bb_min[1]
    zi = np.arange(grid_shape[2]) * spacing + bb_min[2]
    grid = np.stack(np.meshgrid(xi, yi, zi, indexing='ij'), axis=-1).reshape(-1, 3)

    PROTEIN_OCCUPANCY = 1.4 * 2
    protein_tree    = cKDTree(protein_coords)
    dist_protein, _ = protein_tree.query(grid)
    grid            = grid[dist_protein >= PROTEIN_OCCUPANCY]

    pocket_tree    = cKDTree(pocket_coords)
    dist_pocket, _ = pocket_tree.query(grid)

    non_pocket_tree    = cKDTree(non_pocket_coords)
    dist_non_pocket, _ = non_pocket_tree.query(grid)

    ratio = dist_pocket / (dist_non_pocket + 1e-8)
    print(min(ratio), max(ratio), np.mean(ratio), np.std(ratio))

    mask = (dist_pocket < max_dist_from_pocket)
    print(np.sum(mask))
    mask &= (ratio < voronoi_ratio)
    print(np.sum(mask))

    # TODO:
    # Cluster with DBSCAN (perhaps hierarch better? auto connectivity)
    # Filter on top k largest pockets


    return grid[mask]


grid = pocket_to_grid(result['pocket_heavy'], result['protein_heavy'], result['non_pocket_heavy'], padding=6.0, max_dist_from_pocket=6.0, voronoi_ratio=0.6)
print(f'grid points: {len(grid)}')

0.22264932802186435 8.299907662623783 1.8382831615662587 1.1420875662673298
6991
2621
grid points: 2621


In [12]:
import pyrite
from pyrite import Viewer
from pyrite.bounds import Pocket

pocket = Pocket(grid, np.ones_like(grid[:, 0]) * 2.8)

receptor = pyrite.Mol.from_pdb(str(PROTEIN_PATH))

view = Viewer(receptor, pocket)

view

HTML(value='<iframe srcdoc="&lt;div id=&quot;3dmolviewer_1779111445260402&quot;  style=&quot;position: relativ…

In [9]:
pocket.to_pqr('output/{}.pqr'.format(PROTEIN))

In [10]:
pocket_old = Pocket.from_mol(receptor, grid_size=1,neighbors=26, gt_distance_to_outside=14, solvent_accessible=False)

Viewer(receptor, pocket_old)

HTML(value='<iframe srcdoc="&lt;div id=&quot;3dmolviewer_17791113767388842&quot;  style=&quot;position: relati…